In [ ]:
%load_ext autoreload
%autoreload 2

### MS cluster-specific updating

 - Base models generated with `train_mhcprime_with_allele_specific_cluster_holdout_data.py`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "CLSBN26":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

In [ ]:
# imports
from Data.mhc_pseudo_seqs import mhc_seq_dict
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
GPU_ID = 0
# GPU_ID = 1
# GPU_ID = 2
# GPU_ID = 3

# torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 0)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### Base replay set

In [ ]:
# get replay set and process
ms_train_data_for_clustering=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/ms_train_data_for_clustering_replay_set.csv", index_col=0)

ms_train_data_for_clustering=full_process_data(ms_train_data_for_clustering, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data_for_clustering=create_seq_allele_column(ms_train_data_for_clustering)

### PepCL Updating

#### Run PepCL

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
pepcl_params=dict(
            device="cuda",
            steps = 200,            
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            ms_neg_ratio = 1.0,
            num_nonms_pos = 2000,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "logsmoothap",
            nonms_loss_alpha = 10.0,
            ft_lambda = 1.0,
            dr_lambda = 1e-4,
            dr_loss_type = "mse"
)

In [ ]:
# allele specific data
target_allele="A0201"
# target_allele="A0301"
# target_allele="B2705"
# target_allele="B4002"
# target_allele="C0501"
# target_allele="C1202"


allele=target_allele

joint_ext_test_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set.csv", index_col=0)

allele_train_pos_louvain_id_to_seq_allele=load_dict_pickle(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/allele_train_pos_louvain_id_to_seq_allele.pkl")

In [ ]:
%%time
# filter train data down to the allele and length of interest only, not pan-allelic replay
print(allele)
ms_train_data_for_clustering["peptide"]=ms_train_data_for_clustering.seq.str.replace("$", "")
ms_train_data_for_clustering["seq_len"]=ms_train_data_for_clustering.peptide.apply(len)
ms_train_data_for_clustering=create_seq_allele_column(ms_train_data_for_clustering)

length=9

ms_train_data_for_clustering_filtered=ms_train_data_for_clustering.query(f"allele == '{allele}' and seq_len == 9").copy()
print_data_stats(ms_train_data_for_clustering_filtered)

# remove test set overlap
joint_ext_test_df=create_seq_allele_column(joint_ext_test_df)

ms_train_data_for_clustering_filtered_cleaned=remove_test_overlap(ms_train_data_for_clustering_filtered, joint_ext_test_df, seq_allele_col='seq_allele')
print_data_stats(ms_train_data_for_clustering_filtered_cleaned)

In [ ]:
%%time
"""
Representation group updating
"""

allele_specific_clusters={
    'A0201': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 'Full', 'Null'], 
    'A0301': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 'Full', 'Null'], 
    'B2705': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 'Full', 'Null'], 
    'B4002': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 'Full', 'Null'], 
    'C0501': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 'Full', 'Null'], 
    'C1202': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 'Full', 'Null']
}


# get models and data and train
base_and_ft_models={}

allele=target_allele
for lid in allele_specific_clusters[allele]:

    if lid == "Full" or lid == "Null":
        continue

    print("LID", lid)
    louvain_ids=[lid] # match to whatever we are training below

    ms_train_data_for_clustering_filtered_cleaned=add_louvain_flags(train_data_filtered_cleaned, allele_train_pos_louvain_id_to_seq_allele, louvain_ids, seq_allele_col='seq_allele')

    print_data_stats(ms_train_data_for_clustering_filtered_cleaned.query("domain_id == 0"))
    print_data_stats(ms_train_data_for_clustering_filtered_cleaned.query("domain_id == 1"))

    # copy negatives over to domain id 1 as well
    ms_train_data_for_clustering_filtered_cleaned=duplicate_negatives_to_domain1(ms_train_data_for_clustering_filtered_cleaned, allele, label_col='label', allele_col='allele')

    print_data_stats(ms_train_data_for_clustering_filtered_cleaned.query("domain_id == 0"))
    print_data_stats(ms_train_data_for_clustering_filtered_cleaned.query("domain_id == 1"))

    # reset index
    ms_train_data_for_clustering_filtered_cleaned.reset_index(inplace=True, drop=True)

    base_model_name=f"mhcprime_holdout_{allele}_{str(lid)}"
    print("Base model name", base_model_name)

    model_name_ft=f"model_ft_{allele.lower()}_{str(lid)}" 
    model_name_pepcl_1eN4=f"model_pepcl_{allele.lower()}_{str(lid)}_lambda_1eN4" 

    # get base model
    model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Representation_Group_Holdout_Models" / target_allele / "Base"

    mhcprime=init_model(base_model_params, print_check=False, print_params=True)
    mhcprime.load_state_dict(torch.load(f"{model_dir}/{base_model_name}.pt"))

    # get the ft model
    model_ft = copy.deepcopy(mhcprime)
    model_ft_pepcl_1eN4 = copy.deepcopy(mhcprime)

    print("FT")
    pepcl=PepCL(
            model=model_ft,
            train_df=ms_train_data_for_clustering_filtered_cleaned,
            tokenizer=tokenizer,
            use_replay = False,
            **pepcl_params,
    )
    training_stats=pepcl.train()

    print("PepCL")
    pepcl=PepCL(
            model=model_ft_pepcl_1eN4,
            train_df=ms_train_data_for_clustering_filtered_cleaned,
            tokenizer=tokenizer,
            use_replay = True,
            **pepcl_params,

    )
    training_stats=pepcl.train()

    # add to model registry after ft
    # base_and_ft_models[base_model_name]=base_model
    base_and_ft_models[model_name_ft]=model_ft
    base_and_ft_models[model_name_pepcl_1eN4]=model_ft_pepcl_1eN4

In [ ]:
%%time
"""
Save models
"""
# save_models=True
save_models=False

if save_models:
    for model_name, model in base_and_ft_models.items():
        if "pepcl" in model_name:
            model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Representation_Group_Holdout_Models" / target_allele / "PepCL"
        else:
            model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Representation_Group_Holdout_Models" / target_allele / "FT"

        torch.save(model.state_dict(), f"{model_dir}/{model_name}.pt")

#### Lambda controllability test

In [ ]:
# get data for A0201 cluster 1
target_allele="A0201"
joint_ext_test_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set.csv", index_col=0)

allele_train_pos_louvain_id_to_seq_allele=load_dict_pickle(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/allele_train_pos_louvain_id_to_seq_allele.pkl")

In [ ]:
%%time
# filter train data down to the allele and length of interest only
allele=target_allele
print(allele)
ms_train_data_for_clustering["peptide"]=ms_train_data_for_clustering.seq.str.replace("$", "")
ms_train_data_for_clustering["seq_len"]=ms_train_data_for_clustering.peptide.apply(len)
ms_train_data_for_clustering=create_seq_allele_column(ms_train_data_for_clustering)

length=9

ms_train_data_for_clustering_filtered=ms_train_data_for_clustering.query(f"allele == '{allele}' and seq_len == 9").copy()
print_data_stats(ms_train_data_for_clustering_filtered)

# remove test set overlap
joint_ext_test_df=create_seq_allele_column(joint_ext_test_df)

ms_train_data_for_clustering_filtered_cleaned=remove_test_overlap(ms_train_data_for_clustering_filtered, joint_ext_test_df, seq_allele_col='seq_allele')
print_data_stats(ms_train_data_for_clustering_filtered_cleaned)

In [ ]:
# create list of lambdas to rkd suffix (1e-4 are already in there)
lambda_suffix_to_value={
    "0": 0,
    "1eN8": 1e-8,
    "1eN7": 1e-7,
    "1eN6": 1e-6,
    "1eN5": 1e-5,
    "1eN3": 1e-3,
    "1eN2": 1e-2,
}

In [ ]:
# get base model
lid=1
base_model_name=f"mhcprime_holdout_{allele}_{str(lid)}"
print("Base model name", base_model_name)

# get base model
model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Representation_Group_Holdout_Models" / target_allele / "Base"

mhcprime=init_model(base_model_params, print_check=False, print_params=True)
mhcprime.load_state_dict(torch.load(f"{model_dir}/{base_model_name}.pt"))

In [ ]:
%%time
# get models and data and train
base_and_ft_models={}

# target cluster
lid=1

for lambda_suffix, lambda_val in lambda_suffix_to_value.items():
    print("Allele", allele)
    print("LID", lid)
    louvain_ids=[lid] # match to whatever we are training below

    ms_train_data_for_clustering_filtered_cleaned_for_cluster=add_louvain_flags(ms_train_data_for_clustering_filtered_cleaned, allele_train_pos_louvain_id_to_seq_allele, louvain_ids, seq_allele_col='seq_allele')

    print_data_stats(ms_train_data_for_clustering_filtered_cleaned_for_cluster.query("domain_id == 0"))
    print_data_stats(ms_train_data_for_clustering_filtered_cleaned_for_cluster.query("domain_id == 1"))

    # copy negatives over to domain id 1 as well
    ms_train_data_for_clustering_filtered_cleaned_for_cluster=duplicate_negatives_to_domain1(ms_train_data_for_clustering_filtered_cleaned_for_cluster, allele, label_col='label', allele_col='allele')

    print_data_stats(ms_train_data_for_clustering_filtered_cleaned_for_cluster.query("domain_id == 0"))
    print_data_stats(ms_train_data_for_clustering_filtered_cleaned_for_cluster.query("domain_id == 1"))

    # reset index
    ms_train_data_for_clustering_filtered_cleaned_for_cluster.reset_index(inplace=True, drop=True)

    model_name_pepcl=f"model_pepcl_{allele.lower()}_{str(lid)}_lambda_{lambda_suffix}" 

    # get the ft model
    model_pepcl = copy.deepcopy(mhcprime)

    # run updated pepcl function
    pepcl=PepCL(
            model=model_pepcl,
            train_df=ms_train_data_for_clustering_filtered_cleaned_for_cluster,
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,            
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            ms_neg_ratio = 1.0,
            num_nonms_pos = 2000,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            use_replay = True,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "logsmoothap",
            nonms_loss_alpha = 10.0,
            ft_lambda = 1.0,
            dr_lambda = lambda_val,
            dr_loss_type = "mse"
    )

    training_stats=pepcl.train()

    # add to model registry after ft
    base_and_ft_models[model_name_pepcl]=model_pepcl

In [ ]:
%%time
"""
Save models
"""
# save_models=True
save_models=False

if save_models:
    for model_name, model in base_and_ft_models.items():
        if "pepcl" in model_name:
            model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Representation_Group_Holdout_Models" / target_allele / "PepCL"
        else:
            model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Representation_Group_Holdout_Models" / target_allele / "FT"

        torch.save(model.state_dict(), f"{model_dir}/{model_name}.pt")

#### Inference

In [ ]:
# allele specific data
target_allele="A0201"
# target_allele="A0301"
# target_allele="B2705"
# target_allele="B4002"
# target_allele="C0501"
# target_allele="C1202"


allele=target_allele

joint_ext_test_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set.csv", index_col=0)

joint_ext_test_df_resampled=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set_hard_negatives.csv", index_col=0)

In [ ]:
%%time
# joint_ext_test_df
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(joint_ext_test_df, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    joint_ext_test_df[model_name]=output_scores.detach().cpu().tolist()

In [ ]:
%%time
# joint_ext_test_df_resampled
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(joint_ext_test_df_resampled, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    joint_ext_test_df_resampled[model_name]=output_scores.detach().cpu().tolist()

In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
%%time
"""
Save test sets
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    joint_ext_test_df.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set.csv")

    joint_ext_test_df_resampled.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set_hard_negatives.csv")